# NASA FIRMS — tratamento, consolidação e DuckDB

Este notebook lê exclusivamente os CSVs locais baixados por `01_download_firms.ipynb`. Ele não acessa a internet. O processamento das linhas de origem é feito em streaming, sem carregar o conjunto completo em memória.

Produtos: CSV bruto (uma linha por foco), CSV agregado mensal e banco DuckDB com as tabelas `focos_brutos` e `focos_mensais`. A hora de aquisição FIRMS é interpretada como UTC/GMT e gravada sem fuso em `TIMESTAMP`; o nome da coluna documenta a referência GMT.

Uso de IA: ferramentas de IA auxiliaram na elaboração e revisão deste notebook, conforme a declaração do projeto no README.

## 1. Dependências
Instale as bibliotecas caso necessário e reinicie o kernel se o ambiente solicitar.

In [ ]:
# %pip install duckdb

In [ ]:
from pathlib import Path
import csv
import shutil
import sys

CWD = Path.cwd().resolve()
DATASET_DIR = CWD if CWD.name == 'dataset_extractor' else CWD / 'dataset_extractor'
if str(DATASET_DIR) not in sys.path:
    sys.path.insert(0, str(DATASET_DIR))
from firms_pipeline import (AGGREGATE_COLUMNS, RAW_COLUMNS, build_duckdb, transform_sources)

## 2. Configuração e pré-condições
O relatório do download precisa representar uma execução completa. Se estiver em `dry_run`, execute primeiro o download real no notebook 01.

In [ ]:
RAW_DIR = DATASET_DIR / 'csv' / 'raw'
REPORT_PATH = DATASET_DIR / 'csv' / 'download_report.csv'
OUTPUT_DIR = DATASET_DIR / 'output'
RAW_OUTPUT = OUTPUT_DIR / 'focos_brutos_america_sul.csv'
AGGREGATE_OUTPUT = OUTPUT_DIR / 'focos_mensais_america_sul.csv'
DATABASE_PATH = OUTPUT_DIR / 'focos_america_sul.duckdb'

if not REPORT_PATH.exists():
    raise FileNotFoundError(f'Relatório de download não encontrado: {REPORT_PATH}. Execute 01_download_firms.ipynb.')
with REPORT_PATH.open(newline='', encoding='utf-8') as report_file:
    report = list(csv.DictReader(report_file))
if not report:
    raise RuntimeError('Relatório de download vazio; execute o notebook de download.')
bad_statuses = [r for r in report if r['status'] not in {'downloaded', 'skipped'}]
if bad_statuses:
    raise RuntimeError(f"O relatório contém {len(bad_statuses)} entradas não concluídas (dry_run/falhas). Repare os downloads antes do tratamento.")
source_files = []
for item in report:
    path = RAW_DIR / item['source'] / str(item['year']) / item['filename']
    if not path.is_file():
        raise FileNotFoundError(f'Arquivo do relatório ausente: {path}')
    expected, actual = int(item['expected_size']), path.stat().st_size
    if actual != expected:
        raise ValueError(f'Tamanho incorreto em {path}: {actual}; manifesto: {expected}')
    source_files.append(path)
free_bytes = shutil.disk_usage(DATASET_DIR).free
source_bytes = sum(p.stat().st_size for p in source_files)
estimated_required_bytes = 2 * source_bytes  # espaço estimado para bruto e banco durante a geração.
if free_bytes < estimated_required_bytes:
    raise OSError(f'Espaço livre insuficiente: {free_bytes:,} bytes; estimativa mínima: {estimated_required_bytes:,} bytes.')
print(f'Arquivos de origem: {len(source_files):,}; bytes: {source_bytes:,}; espaço livre: {free_bytes:,}')
print(f'Saídas: {RAW_OUTPUT}, {AGGREGATE_OUTPUT}, {DATABASE_PATH}')

## 3. Gerar CSV bruto e agregado
O CSV bruto mantém todos os focos, sem deduplicação. O agregado conta linhas por ano, mês, país, satélite e sensor. Arquivos temporários só são publicados após validação.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result = transform_sources(source_files, RAW_OUTPUT, AGGREGATE_OUTPUT)
print(f"CSV bruto: {result['raw_rows']:,} linhas")
print(f"CSV agregado: {result['aggregate_groups']:,} grupos; soma nfocos={result['nfocos_sum']:,}")
assert result['nfocos_sum'] == result['raw_rows']

### Esquemas esperados
Bruto: `pais,data_hora_gmt,latitude,longitude,satelite,sensor,frp`. Agregado: `ano,mes,pais,satelite,sensor,nfocos`. A normalização aplicada é documentada em `PLANO_NOTEBOOK_FIRMS.md`.

## 4. Criar o DuckDB
O banco é construído em arquivo temporário e substitui o banco final somente após validar contagens, soma de `nfocos` e datas.

In [ ]:
database_result = build_duckdb(RAW_OUTPUT, AGGREGATE_OUTPUT, DATABASE_PATH)
print(f"DuckDB: {DATABASE_PATH} ({database_result['raw_rows']:,} focos; {database_result['monthly_groups']:,} grupos mensais)")
assert database_result['raw_rows'] == database_result['nfocos_sum']

## 5. Executar testes
Os testes completos reconciliam as fontes locais com ambos CSVs e o DuckDB. Eles não acessam a rede. Durante desenvolvimento sem o conjunto integral, rode os testes de fixtures e registre que a reconciliação integral depende dos downloads completos.

In [ ]:
# Execute a partir da raiz do repositório:
# !pytest dataset_extractor/testes -v